In [1]:
import pandas as pd
# 2013–2023
url_2013_2023 = (
    "https://data.cityofchicago.org/api/views/wrvz-psew/rows.csv?accessType=DOWNLOAD"
)
print("Loading 2013–2023 data...")
df_2013_2023 = pd.read_csv(url_2013_2023)
print("Rows 2013–2023:", len(df_2013_2023))
df_2013_2023.head()

Loading 2013–2023 data...


KeyboardInterrupt: 

In [ ]:
dfs = []
years = range(2024, 2027)
limit = 100000
for year in years:
    print(f"\nLoading year {year}...")
    offset = 0
    while True:
        url = (
            "https://data.cityofchicago.org/resource/ajtu-isnz.csv"
            f"?$where=trip_start_timestamp%20between%20'{year}-01-01T00:00:00'"
            f"%20and%20'{year}-12-31T23:59:59'"
            f"&$limit={limit}&$offset={offset}"
        )
        df_chunk = pd.read_csv(url)
        if df_chunk.empty:
            break
        dfs.append(df_chunk)
        offset += limit
        print(f"  Loaded {len(df_chunk)} rows (offset={offset})")
print("\nMerging all data...")
data = pd.concat(dfs, ignore_index=True)
print("Total rows:", len(data))
data.head()



Loading year 2024...
  Loaded 100000 rows (offset=100000)
  Loaded 100000 rows (offset=200000)
  Loaded 100000 rows (offset=300000)
  Loaded 100000 rows (offset=400000)
  Loaded 100000 rows (offset=500000)
  Loaded 100000 rows (offset=600000)
  Loaded 100000 rows (offset=700000)
  Loaded 100000 rows (offset=800000)
  Loaded 100000 rows (offset=900000)
  Loaded 100000 rows (offset=1000000)
  Loaded 100000 rows (offset=1100000)
  Loaded 100000 rows (offset=1200000)
  Loaded 100000 rows (offset=1300000)
  Loaded 100000 rows (offset=1400000)
  Loaded 100000 rows (offset=1500000)


In [ ]:
url = "https://data.cityofchicago.org/resource/ajtu-isnz.csv?$limit=5"
df_sample = pd.read_csv(url)

df_sample.columns

Index(['trip_id', 'taxi_id', 'trip_start_timestamp', 'trip_end_timestamp',
       'trip_seconds', 'trip_miles', 'pickup_census_tract',
       'dropoff_census_tract', 'pickup_community_area',
       'dropoff_community_area', 'fare', 'tips', 'tolls', 'extras',
       'trip_total', 'payment_type', 'company', 'pickup_centroid_latitude',
       'pickup_centroid_longitude', 'pickup_centroid_location',
       'dropoff_centroid_latitude', 'dropoff_centroid_longitude',
       'dropoff_centroid_location'],
      dtype='object')

In [ ]:
url = (
    "https://data.cityofchicago.org/resource/ajtu-isnz.csv"
    "?$select=min(trip_start_timestamp),max(trip_start_timestamp)"
)

pd.read_csv(url)


,min_trip_start_timestamp,max_trip_start_timestamp
0,2024-01-01T00:00:00.000,2026-01-01T00:00:00.000


2024 to Present: Monthly Ride Counts

In [ ]:
import pandas as pd
from urllib.parse import quote

soql = """
SELECT date_trunc_ym(trip_start_timestamp) AS month,
       COUNT(trip_id) AS num_trips
GROUP BY month
ORDER BY month
"""

query = quote(soql)

url = (
    "https://data.cityofchicago.org/resource/ajtu-isnz.csv"
    f"?$query={query}"
)

trips_per_month = pd.read_csv(url)
trips_per_month



,month,num_trips
0,2024-01-01T00:00:00.000,425203
1,2024-02-01T00:00:00.000,440013
2,2024-03-01T00:00:00.000,524134
3,2024-04-01T00:00:00.000,546369
4,2024-05-01T00:00:00.000,619945
5,2024-06-01T00:00:00.000,608266
6,2024-07-01T00:00:00.000,546279
7,2024-08-01T00:00:00.000,563732
8,2024-09-01T00:00:00.000,563224
9,2024-10-01T00:00:00.000,598969


2013 - 2023: Monthly Ride Counts

In [ ]:
# SoQL query: truncate to month, count trips, group and order
soql = """
SELECT date_trunc_ym(trip_start_timestamp) AS month,
       COUNT(trip_id) AS num_trips
GROUP BY month
ORDER BY month
"""

query = quote(soql)

url = (
    "https://data.cityofchicago.org/resource/wrvz-psew.csv"
    f"?$query={query}"
)

trips_per_month_2013_2023 = pd.read_csv(url)
trips_per_month_2013_2023


,month,num_trips
0,2013-01-01T00:00:00.000,1590062
1,2013-02-01T00:00:00.000,1800402
2,2013-03-01T00:00:00.000,2261377
3,2013-04-01T00:00:00.000,2116671
4,2013-05-01T00:00:00.000,2260977
...,...,...
127,2023-08-01T00:00:00.000,553854
128,2023-09-01T00:00:00.000,558682
129,2023-10-01T00:00:00.000,606474
130,2023-11-01T00:00:00.000,527797


In [ ]:
# Plot
plt.figure()
plt.plot(trips_per_month_2013_2023.month, trips_per_month_2013_2023.num_trips, marker='o')
plt.xlabel("Day")
plt.ylabel("Number of trips")
plt.title("Daily trips")
plt.tight_layout()
plt.show()

2024 - Present: Daily Ride Counts

In [ ]:
soql = """
SELECT date_trunc_ymd(trip_start_timestamp) AS day,
       COUNT(trip_id) AS num_trips
GROUP BY day
ORDER BY day
"""

query = quote(soql)

url = (
    "https://data.cityofchicago.org/resource/ajtu-isnz.csv"
    f"?$query={query}"
)

trips_per_day = pd.read_csv(url)
trips_per_day


,day,num_trips
0,2024-01-01T00:00:00.000,9511
1,2024-01-02T00:00:00.000,12807
2,2024-01-03T00:00:00.000,13449
3,2024-01-04T00:00:00.000,13825
4,2024-01-05T00:00:00.000,12953
...,...,...
727,2025-12-28T00:00:00.000,10971
728,2025-12-29T00:00:00.000,13316
729,2025-12-30T00:00:00.000,14947
730,2025-12-31T00:00:00.000,14972


2013 - 2023: Daily Ride Counts

In [ ]:
dfs = []

for year in range(2013, 2024):
    print(f"Loading {year}...")

    soql = f"""
    SELECT date_trunc_ymd(trip_start_timestamp) AS day,
           COUNT(trip_id) AS num_trips
    WHERE trip_start_timestamp >= '{year}-01-01T00:00:00'
      AND trip_start_timestamp <  '{year+1}-01-01T00:00:00'
    GROUP BY day
    ORDER BY day
    """

    query = quote(soql)

    url = (
        "https://data.cityofchicago.org/resource/wrvz-psew.csv"
        f"?$query={query}"
    )

    df_year = pd.read_csv(url)
    dfs.append(df_year)

trips_per_day_2013_2023 = pd.concat(dfs, ignore_index=True)
trips_per_day_2013_2023["day"] = pd.to_datetime(trips_per_day["day"])


Loading 2013...
Loading 2014...
Loading 2015...
Loading 2016...
Loading 2017...
Loading 2018...
Loading 2019...
Loading 2020...
Loading 2021...
Loading 2022...
Loading 2023...


In [ ]:
# trips_per_day_2013_2023 = pd.concat(dfs, ignore_index=True)
# trips_per_day_2013_2023["day"] = pd.to_datetime(trips_per_day["day"])

trips_per_day_2013_2023["day"] = pd.to_datetime(
    trips_per_day["day"],
    errors="coerce"   # converts only truly invalid strings to NaT
)
trips_per_day_2013_2023

,day,num_trips
0,2024-01-01,56953
1,2024-01-02,36830
2,2024-01-03,38808
3,2024-01-04,50391
4,2024-01-05,48354
...,...,...
4012,NaT,13268
4013,NaT,13921
4014,NaT,13529
4015,NaT,11285


In [ ]:
trips_per_day_2013_2023.dtypes


,0
day,datetime64[ns]
num_trips,int64


In [ ]:
trips_per_day_2013_2023.head()

,day,num_trips
0,2024-01-01,56953
1,2024-01-02,36830
2,2024-01-03,38808
3,2024-01-04,50391
4,2024-01-05,48354


In [ ]:
# how many NaT?
trips_per_day["day"].isna().sum()

# show problematic rows
trips_per_day[trips_per_day["day"].isna()].head()


,day,num_trips


In [ ]:
trips_per_day = trips_per_day.dropna(subset=["day"]).reset_index(drop=True)
len(trips_per_day)


732

In [ ]:
import requests
import pandas as pd
from urllib.parse import quote

dfs = []
for year in range(2013, 2024):
    soql = f"""
    SELECT date_trunc_ymd(trip_start_timestamp) AS day,
           COUNT(trip_id) AS num_trips
    WHERE trip_start_timestamp >= '{year}-01-01T00:00:00'
      AND trip_start_timestamp <  '{year+1}-01-01T00:00:00'
    GROUP BY day
    ORDER BY day
    """
    query = quote(soql)
    url = f"https://data.cityofchicago.org/resource/wrvz-psew.json?$query={query}"
    r = requests.get(url)
    df_year = pd.DataFrame(r.json())
    dfs.append(df_year)

trips_per_day = pd.concat(dfs, ignore_index=True)
trips_per_day["day"] = pd.to_datetime(trips_per_day["day"])
trips_per_day["num_trips"] = trips_per_day["num_trips"].astype(int)


## Post-COVID (2022–Present): Hourly Ride Counts

Same source tables as above, but grouped by **day + hour of day** instead of just day, so we can look at intraday demand patterns (e.g. weekday rush hour vs. weekend nights) rather than only day-to-day totals.

In [10]:
import time
import requests
from io import StringIO
from urllib.parse import quote

def fetch_csv_with_retry(url, max_retries=5, timeout=120):
    """GET a CSV URL with retries + exponential backoff for transient
    network errors / timeouts, then load it into a DataFrame."""
    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            resp = requests.get(url, timeout=timeout)
            resp.raise_for_status()
            return pd.read_csv(StringIO(resp.text))
        except Exception as e:
            last_err = e
            wait = 2 ** attempt  # 2, 4, 8, 16, 32 seconds
            print(f"  attempt {attempt}/{max_retries} failed ({e!r}); retrying in {wait}s...")
            time.sleep(wait)
    raise last_err

# 2022-2023 (older table: wrvz-psew)
# Query per MONTH instead of per year -- a full year of GROUP BY day, hour
# is too heavy for Socrata and times out. Also drop ORDER BY (expensive on
# the server for an aggregate query); we sort locally after concatenating.
dfs = []
for year in [2022, 2023]:
    for month in range(1, 13):
        start = f"{year}-{month:02d}-01T00:00:00"
        if month == 12:
            end = f"{year+1}-01-01T00:00:00"
        else:
            end = f"{year}-{month+1:02d}-01T00:00:00"
        print(f"Loading hourly counts for {year}-{month:02d}...")
        soql = f"""
        SELECT date_trunc_ymd(trip_start_timestamp) AS day,
               date_extract_hh(trip_start_timestamp) AS hour,
               COUNT(trip_id) AS num_trips
        WHERE trip_start_timestamp >= '{start}'
          AND trip_start_timestamp <  '{end}'
        GROUP BY day, hour
        LIMIT 5000
        """
        query = quote(soql)
        url = f"https://data.cityofchicago.org/resource/wrvz-psew.csv?$query={query}"
        df_month = fetch_csv_with_retry(url)
        dfs.append(df_month)
        print(f"  -> {len(df_month)} rows")

hourly_22_23 = pd.concat(dfs, ignore_index=True)
hourly_22_23["day"] = pd.to_datetime(hourly_22_23["day"])
hourly_22_23 = hourly_22_23.sort_values(["day", "hour"]).reset_index(drop=True)
print(hourly_22_23.shape)
hourly_22_23.head()


Loading hourly counts for 2022-01...
  -> 744 rows
Loading hourly counts for 2022-02...
  -> 672 rows
Loading hourly counts for 2022-03...
  -> 743 rows
Loading hourly counts for 2022-04...
  -> 720 rows
Loading hourly counts for 2022-05...
  attempt 1/5 failed (ReadTimeout(ReadTimeoutError("HTTPSConnectionPool(host='data.cityofchicago.org', port=443): Read timed out. (read timeout=120)"))); retrying in 2s...
  attempt 2/5 failed (ReadTimeout(ReadTimeoutError("HTTPSConnectionPool(host='data.cityofchicago.org', port=443): Read timed out. (read timeout=120)"))); retrying in 4s...
  attempt 3/5 failed (ReadTimeout(ReadTimeoutError("HTTPSConnectionPool(host='data.cityofchicago.org', port=443): Read timed out. (read timeout=120)"))); retrying in 8s...
  -> 744 rows
Loading hourly counts for 2022-06...
  -> 720 rows
Loading hourly counts for 2022-07...
  -> 744 rows
Loading hourly counts for 2022-08...
  -> 744 rows
Loading hourly counts for 2022-09...
  -> 720 rows
Loading hourly counts for

,day,hour,num_trips
0,2022-01-01,0,458
1,2022-01-01,1,604
2,2022-01-01,2,612
3,2022-01-01,3,433
4,2022-01-01,4,262


In [11]:
# 2024-present (newer table: ajtu-isnz)
# Same month-by-month chunking as above to keep each request light.
import datetime

dfs = []
start_date = datetime.date(2024, 1, 1)
today = datetime.date.today()
cur = start_date
while cur <= today:
    year, month = cur.year, cur.month
    start = f"{year}-{month:02d}-01T00:00:00"
    if month == 12:
        end = f"{year+1}-01-01T00:00:00"
    else:
        end = f"{year}-{month+1:02d}-01T00:00:00"
    print(f"Loading hourly counts for {year}-{month:02d}...")
    soql = f"""
    SELECT date_trunc_ymd(trip_start_timestamp) AS day,
           date_extract_hh(trip_start_timestamp) AS hour,
           COUNT(trip_id) AS num_trips
    WHERE trip_start_timestamp >= '{start}'
      AND trip_start_timestamp <  '{end}'
    GROUP BY day, hour
    LIMIT 5000
    """
    query = quote(soql)
    url = f"https://data.cityofchicago.org/resource/ajtu-isnz.csv?$query={query}"
    df_month = fetch_csv_with_retry(url)
    dfs.append(df_month)
    print(f"  -> {len(df_month)} rows")
    # advance to next month
    if month == 12:
        cur = datetime.date(year + 1, 1, 1)
    else:
        cur = datetime.date(year, month + 1, 1)

hourly_24_26 = pd.concat(dfs, ignore_index=True)
hourly_24_26["day"] = pd.to_datetime(hourly_24_26["day"])
hourly_24_26 = hourly_24_26.sort_values(["day", "hour"]).reset_index(drop=True)
print(hourly_24_26.shape)
hourly_24_26.head()


Loading hourly counts for 2024-01...
  -> 744 rows
Loading hourly counts for 2024-02...
  -> 696 rows
Loading hourly counts for 2024-03...
  -> 743 rows
Loading hourly counts for 2024-04...
  -> 720 rows
Loading hourly counts for 2024-05...
  -> 744 rows
Loading hourly counts for 2024-06...
  -> 720 rows
Loading hourly counts for 2024-07...
  -> 744 rows
Loading hourly counts for 2024-08...
  -> 744 rows
Loading hourly counts for 2024-09...
  -> 720 rows
Loading hourly counts for 2024-10...
  -> 744 rows
Loading hourly counts for 2024-11...
  -> 720 rows
Loading hourly counts for 2024-12...
  -> 744 rows
Loading hourly counts for 2025-01...
  -> 744 rows
Loading hourly counts for 2025-02...
  -> 672 rows
Loading hourly counts for 2025-03...
  -> 743 rows
Loading hourly counts for 2025-04...
  -> 720 rows
Loading hourly counts for 2025-05...
  -> 744 rows
Loading hourly counts for 2025-06...
  -> 720 rows
Loading hourly counts for 2025-07...
  -> 744 rows
Loading hourly counts for 2025-

,day,hour,num_trips
0,2024-01-01,0,462
1,2024-01-01,1,522
2,2024-01-01,2,490
3,2024-01-01,3,269
4,2024-01-01,4,150


In [13]:
import os
os.makedirs("hourly_counts", exist_ok=True)

hourly_22_23.to_csv("hourly_counts/hourly_22_23.csv", index=False)
hourly_24_26.to_csv("hourly_counts/hourly_24_26.csv", index=False)
print("Saved hourly_22_23.csv:", hourly_22_23.shape)
print("Saved hourly_24_26.csv:", hourly_24_26.shape)


Saved hourly_22_23.csv: (17518, 3)
Saved hourly_24_26.csv: (23374, 3)
